In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1423_Jahangirpuri_Delhi_DPCC_1Day.csv")

In [4]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,210.04,363.74,18.48,57.44,45.62,71.16,14.21,1.22,14.39,...,NaN,12.57,82.35,0.18,122.57,0.00,0.00,21.66,988.40,NaN
1,2024-01-02,209.89,347.09,19.32,51.40,43.19,68.27,18.75,1.26,15.41,...,NaN,12.24,78.92,0.19,132.78,0.00,0.00,27.81,987.79,NaN
2,2024-01-03,227.39,378.91,17.57,44.01,37.71,75.85,13.01,1.73,11.76,...,NaN,11.85,90.51,0.24,129.78,0.00,0.00,21.89,987.57,NaN
3,2024-01-04,275.14,450.29,19.76,29.14,31.57,42.54,15.45,1.76,6.91,...,NaN,11.97,89.94,0.50,189.76,0.00,0.00,13.09,987.76,NaN
4,2024-01-05,192.08,355.99,13.35,26.98,25.20,41.67,12.86,1.67,8.98,...,NaN,12.82,93.11,0.39,176.06,0.00,0.00,8.10,987.91,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,144.50,226.29,22.55,38.52,38.80,50.81,13.90,0.82,6.83,...,NaN,14.44,93.43,0.62,176.24,0.40,0.35,10.27,974.76,NaN
362,2024-12-28,118.96,195.38,29.22,32.08,40.85,52.02,13.26,1.15,7.94,...,NaN,14.78,97.09,0.34,190.44,0.06,0.06,16.06,975.00,NaN
363,2024-12-29,119.21,195.61,14.21,23.62,24.08,49.14,14.39,0.67,14.30,...,NaN,14.28,91.66,0.66,203.85,0.00,0.00,45.89,975.05,NaN
364,2024-12-30,130.00,210.08,12.94,25.79,24.22,46.95,15.17,0.79,19.45,...,NaN,13.06,85.92,0.59,203.06,0.00,0.00,43.60,975.45,NaN


In [5]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [6]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 5
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [7]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [8]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (361, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         210.04        363.74       18.48        57.44   
1  2024-01-02         209.89        347.09       19.32        51.40   
2  2024-01-03         227.39        378.91       17.57        44.01   
3  2024-01-04         275.14        450.29       19.76        29.14   
4  2024-01-05         192.08        355.99       13.35        26.98   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      45.62        71.16        14.21        1.22          14.39   
1      43.19        68.27        18.75        1.26          15.41   
2      37.71        75.85        13.01        1.73          11.76   
3      31.57        42.54        15.45        1.76           6.91   
4      25.20        41.67        12.86        1.67           8.98   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.37             2.72    12.57   82.35      0

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [10]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.182356,0.664916,0.032814,0.697354,0.495730,2.097709,-0.380915,-0.615800,-0.870542,-0.154250,0.897148,-1.819013,1.088540,-2.287808,-0.031086,0.0,0.0,-1.343630,1.566062
1,2024-01-02,1.180445,0.532978,0.116800,0.456290,0.372408,1.920097,0.249644,-0.516535,-0.807078,-0.154250,0.793016,-1.862794,0.879868,-2.212059,-2.445860,0.0,0.0,-1.224892,1.453643
2,2024-01-03,1.403424,0.785126,-0.058170,0.161345,0.094299,2.385945,-0.547583,0.649828,-1.034177,-0.207410,1.269045,-1.914535,1.584972,-1.833310,-2.613242,0.0,0.0,-1.339189,1.413098
3,2024-01-04,2.011839,1.350754,0.160792,-0.432135,-0.217305,0.338796,-0.208692,0.724276,-1.335938,-0.260569,1.633504,-1.898615,1.550295,0.136182,0.733295,0.0,0.0,-1.509090,1.448114
4,2024-01-05,0.953516,0.603504,-0.480098,-0.518343,-0.540581,0.285328,-0.568416,0.500930,-1.207145,-0.074511,1.224417,-1.785846,1.743149,-0.697065,-0.031086,0.0,0.0,-1.605432,1.475758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,2024-12-27,0.347267,-0.424263,0.439744,-0.057767,0.149616,0.847049,-0.423971,-1.608449,-1.340916,-0.154250,2.734319,-1.570922,1.762617,1.045178,-0.021043,0.0,0.0,-1.563536,-0.947695
357,2024-12-28,0.021844,-0.669199,1.106630,-0.314796,0.253653,0.921413,-0.512861,-0.789513,-1.271853,3.380845,1.990525,-1.525814,1.985281,-1.075813,0.771235,0.0,0.0,-1.451749,-0.903464
358,2024-12-29,0.025030,-0.667377,-0.394112,-0.652445,-0.597421,0.744415,-0.355915,-1.980692,-0.876141,1.281052,0.666571,-1.592149,1.654935,1.348177,1.519436,0.0,0.0,-0.875822,-0.894250
359,2024-12-30,0.162512,-0.552714,-0.521091,-0.565838,-0.590316,0.609823,-0.247581,-1.682897,-0.555714,1.201313,0.859958,-1.754005,1.305729,0.817929,1.475359,0.0,0.0,-0.920035,-0.820533


In [11]:
df.to_excel('jahangirpuri2024.xlsx', index=False)